# Building and Deploying AutoGluon on Amazon SageMaker

This notebook walks through training an AutoGluon TabularPredictor on Amazon SageMaker
using the official AutoGluon Deep Learning Containers (DLCs), then deploying the model
as a real-time endpoint for inference.

**What you will learn:**
1. Prepare data and upload to S3
2. Write an AutoGluon training script
3. Launch a SageMaker training job with the AutoGluon DLC
4. Deploy the trained model as a SageMaker endpoint
5. Run predictions and clean up resources

**Prerequisites:** An AWS account with SageMaker permissions and `sagemaker` SDK installed.


## 1. Setup and Configuration

In [1]:
# Uncomment to install dependencies if needed
!pip install -U sagemaker==2.256.0 boto3 pandas

In [2]:
import os

import boto3
import pandas as pd
import sagemaker
from sagemaker import image_uris
from sagemaker.estimator import Estimator
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
region = boto3.session.Session().region_name
sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "autogluon-sagemaker-demo"
role = sagemaker.get_execution_role()

print(f"Region: {region}")
print(f"Bucket: s3://{bucket}/{prefix}")
print(f"Role:   {role}")


Region: us-east-1
Bucket: s3://sagemaker-us-east-1-718498700052/autogluon-sagemaker-demo
Role:   arn:aws:iam::718498700052:role/service-role/AmazonSageMaker-ExecutionRole-20240604T205182


## 2. Prepare Data and Upload to S3

We use a sample tabular dataset for binary classification.

In [4]:
# Download sample data (adult income dataset)
train_url = "https://autogluon.s3.amazonaws.com/datasets/Inc/train.csv"
test_url = "https://autogluon.s3.amazonaws.com/datasets/Inc/test.csv"

train_data = pd.read_csv(train_url)
test_data = pd.read_csv(test_url)

label = "class"
print(f"Train shape: {train_data.shape}")
print(f"Test shape:  {test_data.shape}")
train_data.head()


Train shape: (39073, 15)
Test shape:  (9769, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,178478,Bachelors,13,Never-married,Tech-support,Own-child,White,Female,0,0,40,United-States,<=50K
1,23,State-gov,61743,5th-6th,3,Never-married,Transport-moving,Not-in-family,White,Male,0,0,35,United-States,<=50K
2,46,Private,376789,HS-grad,9,Never-married,Other-service,Not-in-family,White,Male,0,0,15,United-States,<=50K
3,55,?,200235,HS-grad,9,Married-civ-spouse,?,Husband,White,Male,0,0,50,United-States,>50K
4,36,Private,224541,7th-8th,4,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,40,El-Salvador,<=50K


In [5]:
# Save locally and upload to S3
os.makedirs("data", exist_ok=True)
train_data.to_csv("data/train.csv", index=False)
test_data.to_csv("data/test.csv", index=False)

train_s3_uri = sess.upload_data(
    path="data/train.csv",
    bucket=bucket,
    key_prefix=f"{prefix}/data/train",
)
test_s3_uri = sess.upload_data(
    path="data/test.csv",
    bucket=bucket,
    key_prefix=f"{prefix}/data/test",
)
print(f"Train data uploaded to: {train_s3_uri}")
print(f"Test data uploaded to:  {test_s3_uri}")


Train data uploaded to: s3://sagemaker-us-east-1-718498700052/autogluon-sagemaker-demo/data/train/train.csv
Test data uploaded to:  s3://sagemaker-us-east-1-718498700052/autogluon-sagemaker-demo/data/test/test.csv


## 3. Write the AutoGluon Training Script

SageMaker runs your training script inside the DLC container. The script reads data
from the input channels, trains a `TabularPredictor`, and saves the model to the
output directory that SageMaker packages as `model.tar.gz`.


In [6]:
%%writefile train.py
import argparse
import os

import pandas as pd
from autogluon.tabular import TabularPredictor


def train():
    parser = argparse.ArgumentParser()
    parser.add_argument("--label", type=str, default="class")
    parser.add_argument("--preset", type=str, default="medium_quality")
    parser.add_argument("--time_limit", type=int, default=120)
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    parser.add_argument("--training", type=str, default=os.environ.get("SM_CHANNEL_TRAINING", "/opt/ml/input/data/training"))
    args, _ = parser.parse_known_args()

    train_files = [f for f in os.listdir(args.training) if f.endswith(".csv")]
    train_path = os.path.join(args.training, train_files[0])
    train_data = pd.read_csv(train_path)
    print(f"Training data shape: {train_data.shape}")

    predictor = TabularPredictor(
        label=args.label,
        path=args.model_dir,
    ).fit(
        train_data=train_data,
        presets=args.preset,
        time_limit=args.time_limit,
    )

    # Save column info for inference
    import json as json_mod
    col_path = os.path.join(args.model_dir, "columns.json")
    cols = [c for c in train_data.columns if c != args.label]
    with open(col_path, "w") as f:
        json_mod.dump(cols, f)

    leaderboard = predictor.leaderboard()
    print(leaderboard)


if __name__ == "__main__":
    train()


Writing train.py


The inference container needs handler functions to load the model and process
requests. The AutoGluon DLC uses TorchServe, which calls `model_fn`, `input_fn`,
`predict_fn`, and `output_fn` in sequence.


In [7]:
%%writefile inference.py
import io
import json
import os

import pandas as pd
from autogluon.tabular import TabularPredictor

# Global model reference
_predictor = None
_columns = None


def model_fn(model_dir):
    """Load the AutoGluon predictor and column metadata."""
    global _predictor, _columns
    _predictor = TabularPredictor.load(model_dir)
    col_path = os.path.join(model_dir, "columns.json")
    if os.path.exists(col_path):
        with open(col_path) as f:
            _columns = json.load(f)
    return _predictor


def input_fn(request_body, request_content_type):
    """Deserialize input data."""
    if request_content_type == "text/csv":
        df = pd.read_csv(io.StringIO(request_body), header=None)
        if _columns and len(df.columns) == len(_columns):
            df.columns = _columns
        return df
    raise ValueError(f"Unsupported content type: {request_content_type}")


def predict_fn(input_data, model):
    """Run prediction."""
    return model.predict(input_data)


def output_fn(prediction, accept):
    """Serialize predictions to JSON."""
    return prediction.to_json(), "application/json"


Writing inference.py


## 4. Retrieve the AutoGluon DLC Image URI

AWS maintains official AutoGluon Deep Learning Containers for training and inference.
We retrieve the image URI for the current region.


In [9]:
# Retrieve the AutoGluon training DLC image URI
# See available images: https://github.com/aws/deep-learning-containers/blob/master/available_images.md
training_image = image_uris.retrieve(
    framework="autogluon",
    region=region,
    version="1.2",  # Update to latest available AutoGluon DLC version
    py_version="py311",
    instance_type="ml.m5.2xlarge",
    image_scope="training",
)
print(f"Training image: {training_image}")


Training image: 763104351884.dkr.ecr.us-east-1.amazonaws.com/autogluon-training:1.2-cpu-py311


## 5. Launch the SageMaker Training Job

We configure a SageMaker `Estimator` with the AutoGluon DLC image, point it at our
training script, and pass hyperparameters that the script reads at runtime.


In [10]:
estimator = Estimator(
    image_uri=training_image,
    entry_point="train.py",
    source_dir=".",  # includes train.py and inference.py
    role=role,
    instance_count=1,
    instance_type="ml.m5.2xlarge",
    sagemaker_session=sess,
    output_path=f"s3://{bucket}/{prefix}/output",
    hyperparameters={
        "label": "class",
        "preset": "medium_quality",
        "time_limit": 120,
    },
    max_run=600,  # max training time in seconds
)

print("Estimator configured.")


Estimator configured.


In [11]:
# Start the training job
estimator.fit(
    inputs={"training": train_s3_uri.rsplit("/", 1)[0]},
    wait=True,
    logs=True,
)

print(f"Training job: {estimator.latest_training_job.name}")
print(f"Model artifact: {estimator.model_data}")


INFO:sagemaker:Creating training-job with name: autogluon-training-2026-03-31-15-36-39-110


2026-03-31 15:36:41 Starting - Starting the training job...
2026-03-31 15:36:58 Starting - Preparing the instances for training...
2026-03-31 15:37:32 Downloading - Downloading the training image......
2026-03-31 15:38:38 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-03-31 15:38:55,879 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-03-31 15:38:55,880 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-03-31 15:38:55,880 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-03-31 15:38:55,888 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-03-31 15:38:55,890 sagemaker_pytorch_container.training INFO     Invoking user training script.
2026-03-31 15:38:57,470 sagemaker-training-toolk

## 6. Deploy the Model as a Real-Time Endpoint

We retrieve the AutoGluon inference DLC and deploy the trained model.


In [12]:
# Retrieve the inference DLC image
inference_image = image_uris.retrieve(
    framework="autogluon",
    region=region,
    version="1.2",  # Match the training version
    py_version="py311",
    instance_type="ml.m5.2xlarge",
    image_scope="inference",
)
print(f"Inference image: {inference_image}")


Inference image: 763104351884.dkr.ecr.us-east-1.amazonaws.com/autogluon-inference:1.2-cpu-py311


In [13]:
model = Model(
    image_uri=inference_image,
    model_data=estimator.model_data,
    source_dir=".",
    entry_point="inference.py",
    role=role,
    sagemaker_session=sess,
)

model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.2xlarge",
)

predictor_sm = Predictor(
    endpoint_name=model.endpoint_name,
    sagemaker_session=sess,
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
)

print(f"Endpoint name: {predictor_sm.endpoint_name}")


INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-east-1-718498700052/autogluon-sagemaker-demo/output/autogluon-training-2026-03-31-15-36-39-110/output/model.tar.gz), script artifact (.), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-east-1-718498700052/autogluon-inference-2026-03-31-15-41-28-447/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: autogluon-inference-2026-03-31-15-48-10-939
INFO:sagemaker:Creating endpoint-config with name autogluon-inference-2026-03-31-15-48-11-696
INFO:sagemaker:Creating endpoint with name autogluon-inference-2026-03-31-15-48-11-696


-------!Endpoint name: autogluon-inference-2026-03-31-15-48-11-696


## 7. Run Predictions

In [14]:
# Prepare test data (drop the label column)
test_no_label = test_data.drop(columns=[label])

# Send a small batch for prediction
sample = test_no_label.head(5)
response = predictor_sm.predict(sample.to_csv(index=False, header=False))

print("Predictions:")
print(response)


Predictions:
{'0': ' <=50K', '1': ' <=50K', '2': ' >50K', '3': ' <=50K', '4': ' <=50K'}


## 8. Clean Up Resources

Delete the endpoint to avoid ongoing charges.


In [15]:
# Delete the endpoint when done
predictor_sm.delete_endpoint(delete_endpoint_config=True)
print("Endpoint deleted.")


INFO:sagemaker:Deleting endpoint configuration with name: autogluon-inference-2026-03-31-15-48-11-696
INFO:sagemaker:Deleting endpoint with name: autogluon-inference-2026-03-31-15-48-11-696


Endpoint deleted.
